# Chapter 25: Transforms & Augmentation

        **Part IV - Feeding Data** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Separate deterministic preprocessing from random augmentation
- Compose tensor transforms
- Apply augmentation only to training data


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## Transforms are callables

A transform receives one sample and returns the transformed sample. Small callables are easy to test and compose.


In [ ]:
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, value):
        for transform in self.transforms:
            value = transform(value)
        return value

class Normalize:
    def __init__(self, mean, std):
        self.mean = torch.as_tensor(mean)[:, None, None]
        self.std = torch.as_tensor(std)[:, None, None]

    def __call__(self, image):
        return (image - self.mean) / self.std

image = torch.rand(3, 8, 8)
normalize = Normalize([0.5] * 3, [0.25] * 3)
print(normalize(image).shape)


## Random augmentation

Random transforms should preserve labels while presenting plausible variations. Here a generator makes the demonstration reproducible.


In [ ]:
class RandomHorizontalFlip:
    def __init__(self, probability=0.5, generator=None):
        self.probability = probability
        self.generator = generator

    def __call__(self, image):
        if torch.rand((), generator=self.generator) < self.probability:
            return image.flip(-1)
        return image

generator = torch.Generator().manual_seed(3)
train_transform = Compose([RandomHorizontalFlip(0.5, generator), normalize])
augmented = train_transform(image)
print("shape preserved:", augmented.shape == image.shape)


## Train and validation differ

Training may be random; validation must be deterministic so scores are comparable across epochs.


In [ ]:
validation_transform = Compose([normalize])
first = validation_transform(image)
second = validation_transform(image)
print("validation deterministic:", torch.equal(first, second))
variants = [train_transform(image) for _ in range(8)]
changed = any(not torch.equal(variants[0], item) for item in variants[1:])
print("training produced more than one variant:", changed)


## Practice

        Work through these without looking back first.

        1. Add random Gaussian noise as a transform.
2. Write a transform that clamps values to [0,1].
3. Explain why validation augmentation would damage measurement.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 25's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
